# 讀寫文件

到目前為止，我們討論了如何處理數據，
以及如何構建、訓練和測試深度學習模型。
然而，有時我們希望保存訓練的模型，
以備將來在各種環境中使用（比如在部署中進行預測）。
此外，當運行一個耗時較長的訓練過程時，
最佳的做法是定期保存中間結果，
以確保在伺服器電源被不小心斷掉時，我們不會損失幾天的計算結果。
因此，現在是時候學習如何加載和存儲權重向量和整個模型了。

## (**加載和保存張量**)

對於單個張量，我們可以直接調用`load`和`save`函數分別讀寫它們。
這兩個函數都要求我們提供一個名稱，`save`要求將要保存的變量作為輸入。


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


我們現在可以將存儲在文件中的數據讀回記憶體。


In [2]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

我們可以[**儲存一個張量列表，然後把它們讀回記憶體。**]


In [3]:
y = torch.zeros(4)
torch.save([x, y],'x-files')
x2, y2 = torch.load('x-files')
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

我們甚至可以(**寫入或讀取從字符串映射到張量的字典**)。
當我們要讀取或寫入模型中的所有權重時，這很方便。


In [4]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

## [**載入和儲存模型參數**]

儲存單個權重向量（或其他張量）確實有用，
但是如果我們想儲存整個模型，並在以後載入它們，
單獨儲存每個向量則會變得很麻煩。
畢竟，我們可能有數百個參數散布在各處。
因此，深度學習框架提供了內建函數來儲存和載入整個網路。
需要注意的一個重要細節是，這將儲存模型的參數而不是儲存整個模型。
例如，如果我們有一個3層多層感知機，我們需要單獨指定架構。
因為模型本身可以包含任意程式碼，所以模型本身難以序列化。
因此，為了恢復模型，我們需要用程式碼生成架構，
然後從磁碟載入參數。
讓我們從熟悉的多層感知機開始嘗試一下。


In [5]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

接下來，我們[**將模型的參數儲存在一個叫做"mlp.params"的檔案中。**]


In [6]:
torch.save(net.state_dict(), 'mlp.params')

為了恢復模型，我們[**實例化了原始多層感知機模型的一個備份。**]
這裡我們不需要隨機初始化模型參數，而是(**直接讀取檔案中儲存的參數。**)


In [7]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

由於兩個實例具有相同的模型參數，在輸入相同的`X`時，
兩個實例的計算結果應該相同。
讓我們來驗證一下。


In [8]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

## 小結

* `save`和`load`函數可用于張量對象的文件讀寫。
* 我們可以通過參數字典保存和加載網路的全部參數。
* 保存架構必須在程式碼中完成，而不是在參數中完成。

## 練習

1. 即使不需要將經過訓練的模型部署到不同的設備上，儲存模型參數還有什麼實際的好處？
1. 假設我們只想復用網路的一部分，以將其合併到不同的網路架構中。比如想在一個新的網路中使用之前網路的前兩層，該怎麼做？
1. 如何同時保存網路架構和參數？需要對架構加上什麼限制？


[Discussions](https://discuss.d2l.ai/t/1839)


練習一：

1. 設計一個接受輸入並計算張量降維的層，它返回$y_k = \sum_{i, j} W_{ijk} x_i x_j$。

我的回答：



以下是實現一個計算張量降維的層：

````python
class TensorReduction(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # 初始化權重張量 W[i,j,k]
        self.W = nn.Parameter(
            torch.randn(in_features, in_features, out_features) / 
            torch.sqrt(torch.tensor(in_features))
        )
    
    def forward(self, x):
        # x shape: (batch_size, in_features)
        # 計算 y_k = sum_{i,j} W_{ijk} x_i x_j
        
        # 方法1：使用einsum（更清晰且高效）
        return torch.einsum('bi,bj,ijk->bk', x, x, self.W)
        
        # 方法2：使用基本操作（更容易理解）
        # xx = torch.bmm(x.unsqueeze(2), x.unsqueeze(1))  # (batch, i, j)
        # return torch.sum(xx.unsqueeze(3) * self.W, dim=(1,2))  # (batch, k)

# 測試代碼
batch_size, in_features, out_features = 2, 3, 4
layer = TensorReduction(in_features, out_features)
x = torch.randn(batch_size, in_features)
y = layer(x)
print(f"輸入形狀: {x.shape}")
print(f"輸出形狀: {y.shape}")
````

主要特點：
1. 使用 einsum 實現高效計算
2. 使用 Xavier 初始化防止梯度消失/爆炸
3. 支持批量處理
4. 提供了兩種實現方式供參考


練習二：

2. 設計一個返回輸入數據的傅立葉係數前半部分的層。

我的回答：





以下是實現一個返回傅立葉係數前半部分的層：

```python
class HalfFFT(nn.Module):
    def __init__(self, norm='ortho'):
        """
        參數：
        norm: 'ortho' 用於正交化FFT結果
        """
        super().__init__()
        self.norm = norm
        
    def forward(self, x):
        # 計算FFT
        fft_result = torch.fft.fft(x, dim=-1, norm=self.norm)
        
        # 只保留前半部分（包括直流分量和正頻率部分）
        half_len = (x.shape[-1] + 1) // 2
        return fft_result[..., :half_len]

# 測試代碼
# 創建一個簡單的正弦波進行測試
t = torch.linspace(0, 1, 16)
x = torch.sin(2 * torch.pi * 2 * t) + torch.sin(2 * torch.pi * 5 * t)
x = x.unsqueeze(0)  # 添加批次維度

layer = HalfFFT()
y = layer(x)

print(f"輸入形狀: {x.shape}")
print(f"輸出形狀: {y.shape}")
print(f"FFT係數:\n{torch.abs(y)}")  # 顯示幅度譜
```


特點：
1. 使用 torch.fft.fft 進行快速傅立葉變換
2. 只保留有用的前半部分係數
3. 支持批量處理
4. 可選的正交化處理

這個層可用於：
- 頻域特徵提取
- 信號處理
- 數據壓縮


練習三：

3. 如何同時保存網路架構和參數？需要對架構加上什麼限制？

我的回答：





以下是同時保存網路架構和參數的方法：

`````python
import torch
import torch.nn as nn

# 1. 定義一個簡單的網路
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size  # 保存架構參數
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        self.hidden = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = torch.relu(self.hidden(x))
        return self.output(x)

# 2. 創建和訓練網路
net = SimpleNet(20, 128, 10)

# 3. 保存完整模型（架構+參數）
torch.save({
    'model_state_dict': net.state_dict(),
    'model_class': SimpleNet,
    'model_args': {
        'input_size': net.input_size,
        'hidden_size': net.hidden_size,
        'output_size': net.output_size
    }
}, 'complete_model.pth')

# 4. 加載模型
def load_complete_model(path):
    checkpoint = torch.load(path)
    model_class = checkpoint['model_class']
    model = model_class(**checkpoint['model_args'])
    model.load_state_dict(checkpoint['model_state_dict'])
    return model

# 測試加載
loaded_net = load_complete_model('complete_model.pth')
`````


對架構的限制：
1. 類必須是可序列化的
2. 初始化參數要明確定義
3. 不能包含閉包或lambda函數
4. 避免使用動態架構

更完整的實現（包含版本控制）：
`````python
class VersionedNet(nn.Module):
    VERSION = "1.0"  # 版本控制
    
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        self.hidden = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = torch.relu(self.hidden(x))
        return self.output(x)
    
    @classmethod
    def load_model(cls, path):
        """加載模型的類方法"""
        checkpoint = torch.load(path)
        
        # 版本檢查
        if checkpoint['version'] != cls.VERSION:
            print(f"Warning: Model version mismatch. Expected {cls.VERSION}, got {checkpoint['version']}")
        
        # 創建模型實例
        model = cls(**checkpoint['model_args'])
        model.load_state_dict(checkpoint['model_state_dict'])
        return model
    
    def save_model(self, path):
        """保存模型的方法"""
        torch.save({
            'version': self.VERSION,
            'model_state_dict': self.state_dict(),
            'model_class': self.__class__,
            'model_args': {
                'input_size': self.input_size,
                'hidden_size': self.hidden_size,
                'output_size': self.output_size
            }
        }, path)

# 使用示例
net = VersionedNet(20, 128, 10)
net.save_model('versioned_model.pth')
loaded_net = VersionedNet.load_model('versioned_model.pth')
`````


最佳實踐：
1. 使用版本控制
2. 保存所有必要的配置參數
3. 提供清晰的加載接口
4. 做好錯誤處理
5. 考慮向後兼容性

這樣可以確保模型可以完整地保存和恢復，同時保持代碼的可維護性。
